# Gás do Povo — DAG para Avaliação de Impacto (v2)

Este notebook constrói o Directed Acyclic Graph (DAG) metodológico para uma **futura avaliação de impacto** do programa Gás do Povo.

**Correções v2 (após auditoria metodológica):**
- Removido o nó "Dados adicionais necessários": em um DAG formal (Pearl 2009), cada nó representa uma variável aleatória do Processo Gerador de Dados (DGP). Disponibilidade de dados é um requisito operacional do pesquisador, não uma causa. A informação foi movida para a nota ao pé do gráfico.
- Adicionados confundidores latentes não observados (*U*): preço/mercado local de GLP e fricções de acesso (alfabetização digital, distância ao revendedor), que afetam tanto o tratamento quanto o desfecho.
- Corrigida a ambiguidade do rótulo "renda" que aparecia em três nós com papéis causais distintos (pré-tratamento, mediador e desfecho).

In [ ]:
!pip install --upgrade matplotlib numpy pandas --quiet

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np

DAG_PATH = "dag_gas_do_povo.png"

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10), dpi=200)
fig.patch.set_facecolor("#F7F8FA")
ax.set_facecolor("#F7F8FA")
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis("off")

pos = {
    "cad":    (2.5,  7.8),
    "gdp":    (8.0,  7.8),
    "out":    (13.5, 7.8),
    "mech":   (10.8, 5.5),
    "u_mkt":  (5.0,  3.5),
    "u_fric": (10.5, 3.5),
}

specs = {
    "cad":    ("X: Elegibilidade Pre-tratamento",
               "CadUnico · renda per capita pre-trat. · BPC · chefia feminina",
               "#DCEBFA", "#3977A8", False),
    "gdp":    ("T: Gas do Povo (voucher)",
               "recebimento e uso do voucher mensal de gas",
               "#DDF3E4", "#378253", False),
    "out":    ("Y: Desfecho Social (pos-programa)",
               "SAN · saude respiratoria · combustivel limpo",
               "#FADDDD", "#B84D4D", False),
    "mech":   ("M: Mecanismos (Mediadores)",
               "acesso a gas · reduc. biomassa · renda domiciliar liberada",
               "#FFF0C9", "#B78320", False),
    "u_mkt":  ("U1: Mercado local de GLP",
               "preco regional · densidade de revendedores credenciados",
               "#F5F0FF", "#7663A8", True),
    "u_fric": ("U2: Friccoes de Acesso",
               "alfab. digital · smartphone · distancia ao revendedor",
               "#F5F0FF", "#7663A8", True),
}


def arrow(a, b, color="#586273", curve=0.0, dashed=False, shrink=78):
    ax.add_patch(FancyArrowPatch(
        pos[a], pos[b],
        arrowstyle="-|>", mutation_scale=16, linewidth=2.0,
        color=color, linestyle="dashed" if dashed else "solid",
        connectionstyle=f"arc3,rad={curve}",
        shrinkA=shrink, shrinkB=shrink, zorder=1,
    ))


arrow("cad",    "gdp",   color="#3977A8")
arrow("cad",    "out",   color="#3977A8", curve=-0.15)
arrow("gdp",    "out",   color="#378253")
arrow("gdp",    "mech",  color="#B78320")
arrow("mech",   "out",   color="#B78320", curve=-0.10)
arrow("u_mkt",  "gdp",   color="#7663A8", dashed=True, curve= 0.15)
arrow("u_mkt",  "out",   color="#7663A8", dashed=True, curve=-0.10)
arrow("u_fric", "gdp",   color="#7663A8", dashed=True, curve=-0.10)

for key, (title, subtitle, fill, edge, latente) in specs.items():
    x, y = pos[key]
    ls = (0, (5, 3)) if latente else "solid"
    ax.add_patch(FancyBboxPatch(
        (x - 2.1, y - 0.56), 4.2, 1.12,
        boxstyle="round,pad=0.06,rounding_size=0.13",
        facecolor=fill, edgecolor=edge, linewidth=1.7,
        linestyle=ls, zorder=3,
    ))
    ax.text(x, y + 0.16, title,    ha="center", va="center",
            fontsize=10.5, fontweight="bold", color="#172033", zorder=4)
    ax.text(x, y - 0.21, subtitle, ha="center", va="center",
            fontsize=8.0, color="#4B5563", zorder=4)

ax.text(7.9, 9.55,
        "DAG v2 — Avaliacao de Impacto do Gas do Povo",
        fontsize=20, fontweight="bold", color="#172033", ha="center")
ax.text(7.9, 9.10,
        "Resultado deve ser externo ao voucher  |  nos U = confundidores latentes (borda tracejada)",
        fontsize=10.5, color="#536074", ha="center")

note = (
    "ANALISE DISPONIVEL AGORA: As bases atuais permitem estudos associativos (cobertura x caracteristicas municipais). "
    "Nao ha resultado social externo nem contrafactual.\n"
    "DADOS NECESSARIOS para identificacao causal: painel municipio x ano com desfecho externo (SAN/saude), "
    "proxy para U1 (preco local GLP) e U2 (acesso digital).\n"
    "QUEBRA ESTRUTURAL: Auxilio Gas (Lei 14.284/2021) != Gas do Povo (Lei 15.348/2026). "
    "Inclua indicador de fase ao combinar as series.\n"
    "CONJUNTO DE AJUSTE (back-door): Z = {renda per capita pre-trat., regiao, BPC, chefia feminina}. "
    "U1 e U2 exigem variaveis instrumentais."
)
ax.text(0.4, 1.9, note, fontsize=9.2, color="#7A4D11", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.6", facecolor="#FFF7E3",
                  edgecolor="#D7A94E", linewidth=1.3))

legend_items = [
    mpatches.Patch(facecolor="#DCEBFA", edgecolor="#3977A8",
                   label="X: Elegibilidade pre-tratamento (observado)"),
    mpatches.Patch(facecolor="#DDF3E4", edgecolor="#378253",
                   label="T: Tratamento — Gas do Povo (observado)"),
    mpatches.Patch(facecolor="#FADDDD", edgecolor="#B84D4D",
                   label="Y: Desfecho social externo (observado)"),
    mpatches.Patch(facecolor="#FFF0C9", edgecolor="#B78320",
                   label="M: Mediadores pos-tratamento (observado)"),
    mpatches.Patch(facecolor="#F5F0FF", edgecolor="#7663A8",
                   label="U: Confundidor latente (nao observado)"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=9,
          framealpha=0.90, edgecolor="#CCCCCC",
          bbox_to_anchor=(0.99, 0.01))

plt.savefig(DAG_PATH, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"DAG v2 salvo em: {DAG_PATH}")

## Interpretação do DAG v2

### Nós e seus papéis causais

| Símbolo | Nó | Papel | Variáveis representadas |
|---------|-----|-------|------------------------|
| **X** | Elegibilidade pré-tratamento | Confundidor observado (back-door) | Renda per capita pré-trat. ≤ ½ SM, inscrição no CadÚnico, BPC, chefia feminina, região |
| **T** | Gás do Povo (voucher) | Tratamento | Recebimento e uso do voucher mensal de gás |
| **Y** | Desfecho social (pós-programa) | Resultado causal de interesse | Segurança Alimentar (SAN), saúde respiratória, combustível limpo |
| **M** | Mecanismos (mediadores) | Mediadores pós-tratamento | Acesso a gás de cozinha, redução de biomassa, renda domiciliar liberada para alimentação |
| **U₁** | Mercado local de GLP | Confundidor **latente** (não observado) | Preço regional do botijão, densidade de revendedores credenciados |
| **U₂** | Fricções de acesso | Confundidor **latente** (não observado) | Alfabetização digital, posse de smartphone, distância ao revendedor |

> **Convensão visual:** bordas sólidas = variáveis observadas (ou observáveis); bordas tracejadas = variáveis latentes/não observadas.

### Caminhos causais e back-door paths

| Caminho | Tipo | Interpretação |
|---------|------|---------------|
| X → T → Y | Back-door direto | X determina elegibilidade; afeta Y independentemente via condições pré-existentes |
| T → Y | Efeito causal total | Caminho de interesse |
| T → M → Y | Efeito indireto (mediado) | Parte do efeito total passa pelos mecanismos |
| U₁ → T, U₁ → Y | Back-door latente | Municípios com GLP mais barato têm maior uptake e melhores desfechos por razões não observadas |
| U₂ → T | Back-door latente | Famílias com menos acesso digital resgatam menos vouchers |

### Conjunto de ajuste (Backdoor Adjustment Set) — sob Ignorabilidade Condicional

Fechar o back-door via **X** requer ajustar por:

```
𝒁 = {renda per capita pré-tratamento, região, BPC, chefia feminina}
```

Isso bloqueia o back-door **X → Y** mas **não** bloqueia os back-doors via U₁ e U₂.  
Para U₁ e U₂ são necessárias variáveis instrumentais (ex.: variação exógena no preço do petróleo para U₁; índice de conectividade municipal para U₂).

### Estratégias de identificação recomendadas

| Desenho | Requisito | Adequado para |
|---------|-----------|---------------|
| **DiD** | Painel pré/pós com municípios comparáveis | Efeito médio do programa (ATE) |
| **PSM** | Variáveis do CadÚnico para escore de propensão | Efeito sobre os tratados (ATT) |
| **RDD** | Corte no limiar de renda per capita (½ SM) | Efeito local (LATE) |
| **IV** | Instrumento exógeno para U₁ ou U₂ | Correção por confundidores latentes |